# Extração, Limpeza e Transformação dos Dados

## Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## Extração dos dados 

Neste estudo, foi utilizado o conjunto de dados consolidado: Dados Acadêmicos - Classificação Racial, Renda, SEXO e Faixa Etária da plataforma Nilo Peçanha (https://moduloextratorpnp.mec.gov.br/pnpquery/5), aqui denominado de Candidatos.csv, para analisar a probabilidade de sucesso (Taxa de Conclusão) de acordo com o perfil demográfico do aluno (Cor, Renda, Idade, Gênero) na Rede Federal.

In [2]:
# Leitura do arquivo
df = pd.read_csv('../Data/Candidatos.csv', sep=';', encoding='utf-8')
df.shape

(244006, 15)

## Limpeza dos dados

Antes de iniciar as análises estatísticas, os dados passaram por processo de preparação e limpeza com as seguintes etapas:

### **Etapa 1:** Padronização das variáveis e observações:

Neste etapa, foram removidos espaços em branco desnecessários (trailing/leading spaces) dos nomes das colunas e dos valores textuais, garantindo consistência na manipulação das strings.

In [3]:
# Limpeza de strings
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

In [4]:
# Renomeação das colunas (variáveis)
df = df.rename(columns={
    'Ano':'ANO', 
    'Região':'REGIAO', 
    'UF':'UF_SIGLA', 
    'Estado':'UF_NOME', 
    'Organização Acadêmica PNP':'ORG_ACAD',
    'Instituicao':'INST', 
    'Instituição (Nome)': 'INST_NOME', 
    'CorRaca':'ETNIA', 
    'RendaFamiliar':'RENDA',
    'FaixaEtaria':'FAIXA_ETARIA',
    'Sexo':'SEXO',
    'Número de concluintes':'CONCLUINTES',
    'Número de ingressantes':'INGRESSANTES',
    'Número de Matrículas':'MATRICULAS',
    'Número de vagas':'VAGAS'
})

### **Etapa 2:** Filtro por Instituição

O estudo focou especificamente nos Institutos Federais de Ensino. Para isso, os dados foram filtrados para incluir apenas as instituições cujos nomes iniciavam com "IF" (Institutos Federais).

In [5]:
# Filtro para Institutos Federais
df = df[df['ORG_ACAD'] == 'Instituto Federal']

print(f"Linhas válidas após manutenção somente dos IFs: {df.shape}")

Linhas válidas após manutenção somente dos IFs: (201550, 15)


### **Etapa 3:** Tratamento de Dados Faltantes e Ruídos

Para evitar que a ausência de declaração distorcesse as análises de perfis demográficos específicos, foram excluídos todos os registros em que as variáveis de interesse (ETNIA, RENDA,FAIXA_ETARIA e SEXO) constavam como S/I (Sem Informação) ou Não Declarada.

In [6]:
# Remoção de classes sem informação declarada
remover = ['S/I', 'Não Declarada', 'Não declarada']
df = df[~df['ETNIA'].isin(remover)]
df = df[~df['RENDA'].isin(remover)]
df = df[~df['FAIXA_ETARIA'].isin(remover)]
df = df[~df['SEXO'].isin(remover)]

# Garante que todas as colunas com contagem de alunos sejam salvas como inteiros
colunas_inteiras = ['CONCLUINTES', 'INGRESSANTES', 'MATRICULAS', 'VAGAS']
for col in colunas_inteiras:
    df[col] = df[col].fillna(0).astype(int)

print(f"Linhas válidas após a deleção das observações sem dados: {df.shape}")

Linhas válidas após a deleção das observações sem dados: (135428, 15)


## Transformação dos dados

### Formatação das observações da variável RENDA

In [7]:
# Dicionário de mapeamento
mapa_renda = {
    '0<RFP<=0,5': 'Muito-Baixa',
    '0,5<RFP<=1': 'Baixa',
    '1<RFP<=1,5': 'Média-Baixa',
    '1,5<RFP<=2,5': 'Média',
    '2,5<RFP<=3,5': 'Média-Alta',
    'RFP>3,5': 'Alta',
    'Não declarada': 'Não-Declarada'
}

# Substituir os valores na coluna do dataframe original
df['RENDA'] = df['RENDA'].map(mapa_renda).fillna(df['RENDA'])


## Gravação do arquivo final de dados

In [8]:
df.to_parquet('../Data/Candidatos_Tratados.parquet', index=False)